In [1]:
# Please execute/shift-return this cell everytime you run the notebook.  Don't edit it. 
%load_ext autoreload
%autoreload 2
from notebook import * 

## FLOPs

In [2]:
! lscpu

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      28
  On-line CPU(s) list:       0-27
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Core(TM) i7-14700K
    CPU family:              6
    Model:                   183
    Thread(s) per core:      2
    Core(s) per socket:      20
    Socket(s):               1
    Stepping:                1
    CPU(s) scaling MHz:      26%
    CPU max MHz:             5600.0000
    CPU min MHz:             800.0000
    BogoMIPS:                6835.20
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush dts acpi mmx fxsr sse 
                             sse2 ss ht tm pbe syscall nx pdpe1gb rdtscp lm cons
                             tant_tsc art arch_perfmon pebs bts rep_go

In [3]:
! nvidia-smi

Tue Sep 29 11:44:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070 ...    Off |   00000000:01:00.0 Off |                  N/A |
| 32%   35C    P8             12W /  100W |      11MiB /  16376MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
! cd metrics; make; echo "algorithm,data_type,size,time,GLOPS" > flops.csv

make: Nothing to be done for 'all'.


In [5]:
! cd ./metrics;  echo -n "tiled_cpu," >> flops.csv
! cd ./metrics; time ./cpumm 2048 512 2>> flops.csv 

Data type size: 4

Time: 16145.062 ms

Throughput: 1.06 GFLOPS


real	0m16.241s
user	0m16.218s
sys	0m0.016s


In [6]:
! cd ./metrics;  echo -n "tiled," >> flops.csv
! cd ./metrics; time ./cudamm 2048 1 2>> flops.csv 
! cd ./metrics;  echo -n "standard," >> flops.csv
! cd ./metrics; time ./cudamm 2048 0 2>> flops.csv 
! cd ./metrics;  echo -n "tiled," >> flops.csv
! cd ./metrics; time ./cudamm 4096 1 2>> flops.csv 
! cd ./metrics;  echo -n "standard," >> flops.csv
! cd ./metrics; time ./cudamm 4096 0 2>> flops.csv 
! cd ./metrics;  echo -n "tiled," >> flops.csv
! cd ./metrics; time ./cudamm 8192 1 2>> flops.csv 
! cd ./metrics;  echo -n "standard," >> flops.csv
! cd ./metrics; time ./cudamm 8192 0 2>> flops.csv 
! cd ./metrics;  echo -n "tiled," >> flops.csv
! cd ./metrics; time ./cudamm_double 4096 1 2>> flops.csv 
! cd ./metrics;  echo -n "tiled," >> flops.csv
! cd ./metrics; time ./cudamm_int 4096 1 2>> flops.csv 
! cd ./metrics;  echo -n "tiled," >> flops.csv
! cd ./metrics; time ./cudamm_double 8192 1 2>> flops.csv 
! cd ./metrics;  echo -n "tiled," >> flops.csv
! cd ./metrics; time ./cudamm_int 8192 1 2>> flops.csv 

Data Type Size: 4
Time elapsed on matrix multiplication of 4 2048x2048 . 2048x2048 on GPU: 12.545184 ms.


Throughput: 1369.44 GFLOPS


real	0m0.339s
user	0m0.099s
sys	0m0.181s
Data Type Size: 4
Time elapsed on matrix multiplication of 4 2048x2048 . 2048x2048 on GPU: 13.779264 ms.


Throughput: 1246.79 GFLOPS


real	0m0.322s
user	0m0.088s
sys	0m0.157s
Data Type Size: 4
Time elapsed on matrix multiplication of 4 4096x4096 . 4096x4096 on GPU: 115.462143 ms.


Throughput: 1190.34 GFLOPS


real	0m0.619s
user	0m0.422s
sys	0m0.176s
Data Type Size: 4
Time elapsed on matrix multiplication of 4 4096x4096 . 4096x4096 on GPU: 154.018585 ms.


Throughput: 892.35 GFLOPS


real	0m0.661s
user	0m0.461s
sys	0m0.173s
Data Type Size: 4
Time elapsed on matrix multiplication of 4 8192x8192 . 8192x8192 on GPU: 1233.573608 ms.


Throughput: 891.32 GFLOPS


real	0m2.736s
user	0m2.410s
sys	0m0.306s
Data Type Size: 4
Time elapsed on matrix multiplication of 4 8192x8192 . 8192x8192 on GPU: 1967.506348 ms.


Thro

In [7]:
display_df_mono(render_csv("./metrics/flops.csv"))

,index,algorithm,data_type,size,time,GLOPS
0,0,tiled_cpu,4,2048x2048x2048,16145.062429,1.060000
1,1,tiled,4,2048x2048x2048,12.545184,1369.440000
2,2,standard,4,2048x2048x2048,13.779264,1246.790000
3,3,tiled,4,4096x4096x4096,115.462143,1190.340000
4,4,standard,4,4096x4096x4096,154.018585,892.350000
5,5,tiled,4,8192x8192x8192,1233.573608,891.320000
6,6,standard,4,8192x8192x8192,1967.506348,558.840000
7,7,tiled,8,4096x4096x4096,333.313843,412.340000
8,8,tiled,4,4096x4096x4096,120.847069,1137.300000
9,9,tiled,8,8192x8192x8192,2653.984863,414.290000


In [8]:
! cd ./metrics;  echo -n "tiled_rtx5090," >> flops.csv
! ssh htseng@kyogre "nvidia-smi; time /nfshome/htseng/courses/CS203/demo/throughput_latency/metrics/cudamm 8192 1 2>> /nfshome/htseng/courses/CS203/demo/throughput_latency/metrics/flops.csv "

Tue Sep 29 11:44:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.43.02              KMD Version: 610.43.02     CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        Off |   00000000:01:00.0 Off |                  N/A |
|  0%   42C    P8             14W /  575W |       2MiB /  32607MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
! cd ./metrics;  echo -n "tiled_rtx5090," >> flops.csv
! ssh htseng@kyogre "time /nfshome/htseng/courses/CS203/demo/throughput_latency/metrics/cudamm 4096 1 2>> /nfshome/htseng/courses/CS203/demo/throughput_latency/metrics/flops.csv "

Data Type Size: 4
Time elapsed on matrix multiplication of 4 4096x4096 . 4096x4096 on GPU: 44.347713 ms.


Throughput: 3099.12 GFLOPS


real	0m0.588s
user	0m0.361s
sys	0m0.204s


In [10]:
display_df_mono(render_csv("./metrics/flops.csv"))

,index,algorithm,data_type,size,time,GLOPS
0,0,tiled_cpu,4,2048x2048x2048,16145.062429,1.060000
1,1,tiled,4,2048x2048x2048,12.545184,1369.440000
2,2,standard,4,2048x2048x2048,13.779264,1246.790000
3,3,tiled,4,4096x4096x4096,115.462143,1190.340000
4,4,standard,4,4096x4096x4096,154.018585,892.350000
5,5,tiled,4,8192x8192x8192,1233.573608,891.320000
6,6,standard,4,8192x8192x8192,1967.506348,558.840000
7,7,tiled,8,4096x4096x4096,333.313843,412.340000
8,8,tiled,4,4096x4096x4096,120.847069,1137.300000
9,9,tiled,8,8192x8192x8192,2653.984863,414.290000


In [11]:
FLOPS_Ratio = 3177.570000/925.890000
print(FLOPS_Ratio)

3.431908758059813


In [12]:
Speedup = 1187.524780/346.022278	
print(Speedup)

3.431931570602515


## Does the FLOPS translate to other applications?

In [13]:
! make -C gpusort clean; make -C gpusort bitonic CFLAGS=-DBLOCKS=8192

make: Entering directory '/nfshome/htseng/courses/CS203/demo/throughput_latency/gpusort'
rm -f	*.o hybridsort hybridsort_cpu bitonic
make: Leaving directory '/nfshome/htseng/courses/CS203/demo/throughput_latency/gpusort'
make: Entering directory '/nfshome/htseng/courses/CS203/demo/throughput_latency/gpusort'
/usr/local/cuda/bin/nvcc -O3 -DBLOCKS=8192 bitonic.cu -o bitonic
nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
make: Leaving directory '/nfshome/htseng/courses/CS203/demo/throughput_latency/gpusort'


In [14]:
! time ./gpusort/bitonic

Sorting 8388608 numbers
Elapsed time: 0.129s
SORTING SUCCESSFUL

real	0m0.393s
user	0m0.103s
sys	0m0.155s


In [15]:
! ssh htseng@lapras "cd /nfshome/htseng/courses/CS203/demo/throughput_latency/; make -C gpusort clean; make -C gpusort bitonic CFLAGS=-DBLOCKS=8192; time ./gpusort/bitonic"

make: Entering directory '/nfshome/htseng/courses/CS203/demo/throughput_latency/gpusort'
rm -f	*.o hybridsort hybridsort_cpu bitonic
make: Leaving directory '/nfshome/htseng/courses/CS203/demo/throughput_latency/gpusort'
make: Entering directory '/nfshome/htseng/courses/CS203/demo/throughput_latency/gpusort'
/usr/local/cuda/bin/nvcc -O3 -DBLOCKS=8192 bitonic.cu -o bitonic
make: Leaving directory '/nfshome/htseng/courses/CS203/demo/throughput_latency/gpusort'
Sorting 8388608 numbers
Elapsed time: 0.148s
SORTING SUCCESSFUL

real	0m0.293s
user	0m0.104s
sys	0m0.183s
